# LLigMa Small — training on a Kaggle GPU

Trains the dense 0.39B preset (`configs/lligma_3_0_small.yaml`, 386,721,792 parameters with a 32K vocabulary) on a Kaggle GPU.

**Prerequisites**

1. Notebook settings -> Accelerator: **GPU** (P100 or T4). Not TPU: the TPU runtime needs torch_xla or JAX, while this code is plain PyTorch eager.
2. Notebook settings -> Internet: **On**, only if you clone the repository or download data inside the notebook.
3. Attach the repository as a Kaggle dataset (Add Data), or set `REPO_URL` in the third cell.

**Session model**

- Kaggle GPU sessions last at most **12 hours**, with roughly **30 GPU hours per week**.
- Nothing survives a session unless it is written under `/kaggle/working` and the notebook version is saved.
- Training writes a resumable checkpoint every 500 optimizer steps; the next session picks up the newest one automatically.

In [ ]:
import torch

if not torch.cuda.is_available():
    print('No GPU visible: set Accelerator = GPU (P100 or T4) in Notebook settings.')
    print('On a TPU runtime torch.cuda is empty by design; this notebook targets GPU.')
else:
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0))
    print('Total memory: %.1f GB' % (props.total_memory / 1e9))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
    print('torch:', torch.__version__)

In [ ]:
from pathlib import Path
import subprocess
import sys

# If you prefer cloning instead of attaching a dataset, set REPO_URL and enable Internet.
REPO_URL = ''

def looks_like_project(path):
    return (path / 'lligma').is_dir() and (path / 'scripts' / 'train.py').is_file()

PROJECT_DIR = None
input_dir = Path('/kaggle/input')
if input_dir.is_dir():
    for candidate in sorted(input_dir.iterdir()):
        if not candidate.is_dir():
            continue
        if looks_like_project(candidate):
            PROJECT_DIR = candidate
            break
        for nested in sorted(candidate.iterdir()):
            if nested.is_dir() and looks_like_project(nested):
                PROJECT_DIR = nested
                break
        if PROJECT_DIR is not None:
            break

if PROJECT_DIR is None:
    if not REPO_URL:
        raise RuntimeError('Project not found under /kaggle/input. Attach the repository as a Kaggle dataset (Add Data) or set REPO_URL above and enable Internet.')
    target = Path('/kaggle/working/lligma-repo')
    if not target.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(target)], check=True)
    PROJECT_DIR = target

sys.path.insert(0, str(PROJECT_DIR))
import lligma
print('Project root:', PROJECT_DIR)
print('lligma loaded from:', lligma.__file__)

CONFIG_PATH = PROJECT_DIR / 'configs' / 'lligma_3_0_small.yaml'
assert CONFIG_PATH.is_file(), 'Missing config: %s' % CONFIG_PATH
print('Config:', CONFIG_PATH)

In [ ]:
import subprocess
import sys

requirements = (PROJECT_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
# The Kaggle image already ships a CUDA build of torch: reinstalling it would replace it.
packages = [line.strip() for line in requirements if line.strip() and not line.strip().lower().startswith('torch')]
print('Installing:', packages)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies ready')

In [ ]:
from pathlib import Path
import subprocess
import sys

WORK_DIR = Path('/kaggle/working')
# Attached datasets appear under /kaggle/input/<dataset-name>/. This searches for
# a train*.parquet anywhere there; set DATA_DIR by hand if your layout differs.
DATA_DIR = None
INPUT_ROOT = Path('/kaggle/input')
if INPUT_ROOT.is_dir():
    matches = sorted(INPUT_ROOT.rglob('train*.parquet'))
    if matches:
        DATA_DIR = matches[0].parent
if DATA_DIR is None:
    DATA_DIR = WORK_DIR / 'data'

TOKENIZER_DIR = WORK_DIR / 'tokenizer-32k'
OUT_DIR = WORK_DIR / 'runs' / 'lligma-small'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_matches = sorted(DATA_DIR.glob('train*.parquet'))
valid_matches = sorted(DATA_DIR.glob('valid*.parquet'))
assert train_matches, 'No train*.parquet under %s. Attach a dataset with Add Data, or set DATA_DIR.' % DATA_DIR
assert valid_matches, 'No valid*.parquet under %s. It must sit next to the training file.' % DATA_DIR
TRAIN_FILE = train_matches[0]
VALIDATION_FILE = valid_matches[0]
print('Data dir:', DATA_DIR)
print('Train file:', TRAIN_FILE)
print('Validation file:', VALIDATION_FILE)

if (TOKENIZER_DIR / 'tokenizer.json').is_file():
    print('Tokenizer already present:', TOKENIZER_DIR)
else:
    cmd = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_tokenizer.py'),
           '--input', str(TRAIN_FILE),
           '--vocab-size', '32768',
           '--output-dir', str(TOKENIZER_DIR)]
    print('Training tokenizer:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
import subprocess
import sys
from pathlib import Path
import torch

MAX_STEPS = 20000

def checkpoint_step(path):
    try:
        return int(path.stem.split('-')[-1])
    except ValueError:
        return -1

roots = [Path('/kaggle/working')]
if Path('/kaggle/input').is_dir():
    roots.append(Path('/kaggle/input'))
checkpoints = []
for root in roots:
    checkpoints.extend(root.rglob('checkpoint-*.pt'))
resume = max(checkpoints, key=checkpoint_step) if checkpoints else None
resume_step = checkpoint_step(resume) if resume else 0
print('Latest checkpoint:', resume, 'at step', resume_step)

if resume_step >= MAX_STEPS:
    print('Training already reached step %d. Raise MAX_STEPS to continue.' % resume_step)
else:
    cmd = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train.py'),
           '--config', str(PROJECT_DIR / 'configs' / 'lligma_3_0_small.yaml'),
           '--train-file', str(TRAIN_FILE),
           '--validation-file', str(VALIDATION_FILE),
           '--tokenizer', str(TOKENIZER_DIR),
           '--output-dir', str(OUT_DIR),
           '--sequence-length', '1024',
           '--batch-size', '4',
           '--gradient-accumulation-steps', '8',
           '--max-steps', str(MAX_STEPS),
           '--eval-steps', '500',
           '--save-every', '500',
           '--device', 'cuda']
    if torch.cuda.is_bf16_supported():
        cmd.append('--bf16')
    else:
        print('bf16 not supported on this GPU: training in fp32')
    if resume is not None and resume_step > 0:
        cmd.extend(['--resume-from', str(resume)])
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd)
    print('Exit code:', result.returncode)
    if result.returncode != 0:
        raise RuntimeError('Training failed, see the log above')

## Persisting the run

Only `/kaggle/working` is kept, and only when you save a notebook version (`Save Version -> Save & Run All`). Output is capped at **20 GB per notebook**, so keep the last checkpoints instead of every one.

In the next session, attach that saved output with **Add Data -> Your Work**: the resume cell searches `/kaggle/input` as well and continues from the newest checkpoint.

In [ ]:
from pathlib import Path

OUT_DIR = Path('/kaggle/working/runs/lligma-small')
checkpoints = sorted(OUT_DIR.glob('checkpoint-*.pt'))
print('Checkpoints in the output folder:', len(checkpoints))
for path in checkpoints[-3:]:
    print(' ', path.name, '%.2f GB' % (path.stat().st_size / 1e9))
print()
print('To continue in a new session:')
print('1. Save Version -> Save & Run All so /kaggle/working is kept.')
print('2. Next session: Add Data -> Your Work, attach that output.')
print('3. Rerun the cells: the training cell resumes from the newest checkpoint found.')
print()
print('Model weights about to be written by save_pretrained:')
print('  fp32 ~1.6 GB, bf16 ~0.8 GB for 0.39B parameters')